In [1]:
import pandas as pd
import requests
import time
import pyarrow
from binance.client import Client

In [2]:
def download_klines(
    symbol: str,
    interval: str,
    start_date: str,
    end_date: str
) -> pd.DataFrame:

    client = Client()

    klines = client.get_historical_klines(
        symbol=symbol,
        interval=interval,
        start_str=start_date,
        end_str=end_date
    )
    print(f"Download complete. Received {len(klines)} raw candles.")

    columns = [
        'timestamp', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_asset_volume', 'trades',
        'taker_buy_base', 'taker_buy_quote', 'ignore'
    ]

    df = pd.DataFrame(klines, columns=columns)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms").astype("datetime64[ns]")
    df = df.sort_values("timestamp").reset_index(drop=True)

    numeric_cols = ["open", "high", "low", "close", "volume", "trades",
                    "quote_asset_volume", "taker_buy_base", "taker_buy_quote"]
    df[numeric_cols] = df[numeric_cols].astype(float)
    df = df[["timestamp"] + numeric_cols].copy()
    return df

In [3]:
data = download_klines(symbol = 'BTCUSDT',
    interval = Client.KLINE_INTERVAL_5MINUTE,
    start_date = "1 Feb, 2021",
    end_date = "4 Sept, 2026")

Download complete. Received 587596 raw candles.


In [4]:
numeric_cols = ["open", "high", "low", "close", "volume", "trades",
                "quote_asset_volume", "taker_buy_base", "taker_buy_quote"]

assert all(data[col].dtype == float for col in numeric_cols), "Not all numeric columns are float dtype"
print("✅ validation passed: all necessary columns are float type")
assert data.isna().sum().sum() == 0, f"Missing values found in columns: {missing[missing > 0].to_dict()}"
print("✅ validation passed: dataset doesn't contain null nor missing values")
assert pd.api.types.is_datetime64_ns_dtype(data["timestamp"]), f"timestamp column is not datetime64[ns], got {data['timestamp'].dtype}"
print("✅ validation passed: timestamp is datatime variable")

✅ validation passed: all necessary columns are float type
✅ validation passed: dataset doesn't contain null nor missing values
✅ validation passed: timestamp is datatime variable


In [7]:
data.head()

,timestamp,open,high,low,close,volume,trades,quote_asset_volume,taker_buy_base,taker_buy_quote
0,2021-02-01 00:00:00,33092.97,33106.33,32777.14,32869.04,487.613325,12171.0,1.606026e+07,179.774943,5.918124e+06
1,2021-02-01 00:05:00,32866.41,32868.46,32464.24,32576.60,727.310947,14776.0,2.373056e+07,283.236278,9.238524e+06
2,2021-02-01 00:10:00,32580.67,32663.48,32545.02,32591.86,373.355815,8163.0,1.217302e+07,139.934790,4.562684e+06
3,2021-02-01 00:15:00,32591.87,32646.40,32300.00,32466.21,554.782922,11846.0,1.800890e+07,299.117644,9.710702e+06
4,2021-02-01 00:20:00,32466.20,32574.30,32330.64,32426.49,439.945032,10034.0,1.427537e+07,246.950888,8.012313e+06


In [5]:
data.to_parquet('data_binance.parquet', index=False)